# Welcome to NeuroMorph!

NeuroMorph is a deep learning pipeline that generates cortical segmentation and cortical reconstructions with high speed and accuracy.





Below is a schema of how NeuroMorph is organised

![NeuroMorph_Schema](<./media/NeuroMorph_Schema.png>)

| Input Parameter | Requirements |
|---|---|
|Modality	| T1-weighted MRI
|Format	NIfTI | (.nii, .nii.gz)
|Resolution	| 1mm isotropic
|Orientation |	LIA recommended; the pipeline will attempt to reorient automatically
|Preprocessing	| Internal preprocessing: shape, size and orientation conform only. No other preprocessing is required e.g., skull stripping, bias field correction, etc.|
|Space	| Any (No registration needed) | 

# Quickstart Inference

If you want to generate the outputs as quickly as possible with your own data, you can run the cell below utilising docker container. Make sure to replace the input and output paths with your own.


In [ ]:
# Generate default outputs for a single file
%%bash
docker pull rocknroll87q/neuromorph:latest
docker run --rm -v /path/to/data:/input -v /path/to/output_dir:/output rocknroll87q/neuromorph \
    --data.vol_in=/input/scan.nii.gz --data.output_dir=/output/

In [ ]:
# Generate default outputs for all files (containing .nii.gz extension) in the input directory
%%bash
docker pull rocknroll87q/neuromorph:latest
docker run --rm -v /path/to/data:/input -v /path/to/output_dir:/output rocknroll87q/neuromorph \
    --data.vol_in=/input/ --data.output_dir=/output/ --input_output.no_of_save_outputs=-1 


# Inference Examples

Follow the steps below to run different types of inference on real example MRI data from the AHEAD dataset or synthetic data examples

See the &nbsp; 📓 [Installation Guide](./docs/installation.md) for instructions on the different ways to run the inference pipeline. The following examples assume you have followed the pip, miniforge, or docker installation instructions. To run via the docker or singularity container, simply update the run command used with any different arg options you wish to use. 

In [ ]:
from pathlib import Path
import sys
import os
os.chdir(globals()['_dh'][0])

IN_DIR = Path('./tmp/neuromorph_input')
OUT_ROOT = Path('./tmp/neuromorph_output')
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)


## Generate Examples

In [ ]:
!{sys.executable} ./scripts/get_test_data.py --output {IN_DIR} -n 2

## Single Volume Test

In [ ]:
!{sys.executable}  ./scripts/run_inference.py \
    --data.output_dir='{OUT_ROOT}' \
    --data.vol_in='{IN_DIR}/sub-0001_T1w.nii.gz'

## Multiple Volume Test

To process an entire directory of volumes, we can simply set the `--data.vol_in` argument to the directory containing the volumes. The script will automatically find all `.nii.gz` files in that directory and process them one by one.


In [ ]:
!{sys.executable} ./scripts/run_inference.py \
         --data.vol_in='./tmp/neuromorph_input/' \
         --data.output_dir='./tmp/neuromorph_output' \
         --input_output.no_of_save_outputs=-1 


Alternatively, we can set the parameter `--data.file_identifier` to a substring that can be used to find the T1w files when recursively searching a data.vol_in directory. For example, if we set `--data.file_identifier='T1w'`, the script will look for all files containing 'T1w' in their names.

## Inference From CSV

We can process a list of volumes specified in a CSV file. The CSV file should have a column named 'T1' that contains the paths to the input volumes. Paths need to be relative if using a singularity container e.g. "/NeuroMorph/data/". A "Subject" column is also needed in order to correctly identify each subject. 

In [ ]:
!{sys.executable} ./scripts/run_inference.py \
         --data.Path_in_csv='./tmp/neuromorph_input/' \
         --data.Filename_csv='subjects.csv' \
         --data.output_dir='./tmp/neuromorph_output/' 

# Saving Options

The quantitative morphological features for each subject will always be saved to a CSV file in the output directory. 

You can optionally save intermediate objects such as the cortical segementation and surfacce reconstruction using the following parameters:


Set any of the following to `True`/`False` to flag an output.

| Flag | Description | Notes |
|---|---|---|
| `--input_output.no_of_save_outputs` | Number of subjects to save | |
| `--input_output.out_T1`   | Conformed T1w volume used for models | |
| `--input_output.out_segmentation`   | Cortical segmentation volume | |
| `--input_output.out_surface_type`   | Output WM, Pial, or both surfaces | |
| `--input_output.out_surface_mesh`   | Cortical surface reconstruction (cortical thickness) | Saved as a `.ply` mesh with CTh overlay by default |
| `--input_output.out_curvature_mesh` | Cortical surface reconstruction (curvature) | Saved as a `.ply` mesh with curvature values overlay |
| `--input_output.out_t1w_overlay_mesh`   | Cortical surface reconstruction (T1w overlay) | Saved as a `.ply` mesh with interpolated T1w intensity values overlay |
| `--input_output.out_surface_fs` | Cortical surface reconstruction (FreeSurfer format) | Saves the selected surface reconstruction as a `.surf` file, with an additional `.thickness` file for loading into FreeSurfer |
| `--input_output.out_level_set` | ImplicitNet predicted level set | Saves the level set predicted by the ImplicitNet model |
| `--input_output.out_distance_set` | ImplicitNet predicted distance set | Saves the distance set predicted by the ImplicitNet model |

# Advanced Options

## Surface Reconstruction Calibration

The parameters below are recommended to remain at their defaults for best results. They are provided in case the user wishes to alter the final surface reconstruction. See the manuscript for further detail on any of these parameters.

#### Marching Cubes Level

`testing.pial_mc_level`
`testing.wm_mc_level`

These parameters alter the level of the marching cubes algorithm for each mesh type, effectively cutting at different depths into the tissue when generating the initial unprocessed mesh. Increasing the mc_level results in deeper sulci and smaller gyri, and vice versa. We suggest keeping the default parameters, which were rigorously optimised.

#### Normal Multiplier

`testing.pial_norm_multiplier`
`testing.wm_norm_multiplier`

Altering the norm_multiplier inflates the unprocessed surface, moving vertices along their face normals to retain cortical geometry. This is primarily used to recover the effects of mc_level, but can also be used for other purposes.

#### Mesh Decimation Target

`testing.mesh_decimation_target`

The default number of vertices is set to 7500. We found this to be the minimum number at which the cortex retains its geometry while reducing redundancy of vertices per face.

#### Mesh Post Processing

`testing.apply_mesh_post_processing`

To improve mesh quality and reduce the number of intersections, post processing can optionally be applied to the mesh. This is achieved by applying

* Minor Taubin smoothing to the whole mesh
* Laplacian smoothing on intersecting faces
* Up to 5 rounds of Taubin smoothing on intersecting faces

## Compute and Runtime Settings

#### Keep Mesh in Memory

`keep_mesh_in_memory`

When running inference or model testing, predicted meshes can be kept in memory after parallel processing if they are needed for further analysis, or removed from memory if they are only generated for metrics. Meshes are still saved to disk regardless of this setting.

#### Limit CPU Count

`limit_cpu_count`

Limits the number of CPU cores used during inference or model testing. If set to None, all available cores are used minus one. If you find that running one of the cells in the inference guide is finished but there is no output, it may be a CPU crash. In this case, try setting the limit_cpu_count to a lower number of cores. 

You can confirm if a CPU crashed while running a cell in the inference guide by checking the exit code:
```python
print(_exit_code)
```
If the exit code is -9, this indicates a CPU crash.



#### Device

`device`

Specifies the device used for inference. Options are 'auto', 'cpu' or 'gpu'. If set to 'auto', the script uses the GPU when available, otherwise it falls back to the CPU.